In [0]:
from pyspark.sql import functions as F

SILVER = "workspace.s4lake_silver"
GOLD = "workspace.s4lake_gold"
DATA_CORTE = "2026-09-22"

def salvar(df, schema, tabela):
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{schema}.{tabela}")

In [0]:
fato_titulos = spark.table(f"{GOLD}.fato_titulos")
clientes = spark.table(f"{SILVER}.clientes")

clientes_dim = clientes.select(
    "cod_cliente",
    "ramo_atividade",
    "ramo_descricao",
    "uf",
    F.col("cond_pagamento").alias("cond_pagamento_cadastro")
)

titulos_dim = fato_titulos.join(clientes_dim, on="cod_cliente", how="left")

In [0]:
print("linhas:", titulos_dim.count())                                                  # 28725
print("ramo nulo:", titulos_dim.filter(F.col("ramo_atividade").isNull()).count())      # 0
print("uf nula:", titulos_dim.filter(F.col("uf").isNull()).count())                    # 0
print("prazo divergente:", titulos_dim.filter(
    F.col("cond_pagamento") != F.col("cond_pagamento_cadastro")).count())              # 0
print("prazo título nulo:", titulos_dim.filter(F.col("cond_pagamento").isNull()).count())             # 0
print("prazo cadastro nulo:", titulos_dim.filter(F.col("cond_pagamento_cadastro").isNull()).count())  # 0

In [0]:
def calcular_kpis(df, colunas_grupo):
    corte = F.lit(DATA_CORTE).cast("date")

    aberto = F.col("status") == "aberto"            # CONFIRA: valores exatos da coluna status
    pago = F.col("status") == "pago"
    vencido = aberto & (F.col("dias_atraso") > 0)
    pago_com_atraso = pago & (F.col("dias_atraso") > 0)
    atraso_positivo = F.greatest(F.col("dias_atraso"), F.lit(0))
    # CONFIRA: use a mesma coluna de data que o notebook 07 usou no faturamento_90d
    no_periodo = F.col("data_documento").between(F.date_sub(corte, 89), corte)

    somas = df.groupBy(*colunas_grupo).agg(
        F.count(F.lit(1)).alias("qtd_titulos"),
        F.countDistinct("cod_cliente").alias("qtd_clientes"),
        F.sum(F.when(aberto, F.col("valor"))).alias("carteira_aberta"),
        F.sum(F.when(vencido, F.col("valor"))).alias("carteira_vencida"),
        F.sum(F.when(pago, F.lit(1))).alias("qtd_pagos"),
        F.sum(F.when(pago_com_atraso, F.lit(1))).alias("qtd_pagos_com_atraso"),
        F.sum(F.when(pago, atraso_positivo)).alias("soma_atraso"),
        F.sum(F.when(pago, F.col("valor"))).alias("soma_valor_pagos"),
        F.sum(F.when(pago, F.col("valor") * atraso_positivo)).alias("soma_valor_x_atraso"),
        F.sum(F.when(no_periodo, F.col("valor"))).alias("faturamento_90d"),
        F.sum(F.when(no_periodo, F.col("valor") * F.col("prazo_dias"))).alias("soma_valor_x_prazo_90d"),
    )

    return (
        somas
        .withColumn("pct_vencido",
                    F.round(F.try_divide(F.col("carteira_vencida"), F.col("carteira_aberta")) * 100, 2))
        .withColumn("pct_pagos_com_atraso",
                    F.round(F.try_divide(F.col("qtd_pagos_com_atraso"), F.col("qtd_pagos")) * 100, 2))
        .withColumn("atraso_medio_simples",
                    F.round(F.try_divide(F.col("soma_atraso"), F.col("qtd_pagos")), 2))
        .withColumn("atraso_medio_ponderado",
                    F.round(F.try_divide(F.col("soma_valor_x_atraso"), F.col("soma_valor_pagos")), 2))
        .withColumn("dso",
                    F.round(F.try_divide(F.col("carteira_aberta"), F.col("faturamento_90d")) * 90, 2))
        .withColumn("prazo_medio_ponderado",
                    F.round(F.try_divide(F.col("soma_valor_x_prazo_90d"), F.col("faturamento_90d")), 2))
        .withColumn("dias_alem_prazo",
                    F.round(F.col("dso") - F.col("prazo_medio_ponderado"), 2))
        .withColumn("data_corte", corte)
    )

In [0]:
teste = calcular_kpis(titulos_dim, []).first()
gerais = spark.table(f"{GOLD}.kpis_gerais").first()

colunas = ["carteira_aberta", "carteira_vencida", "qtd_pagos", "qtd_pagos_com_atraso",
           "soma_valor_pagos", "faturamento_90d", "pct_vencido", "pct_pagos_com_atraso",
           "atraso_medio_simples", "atraso_medio_ponderado", "dso", "prazo_medio_ponderado"]

for c in colunas:
    a, b = float(teste[c]), float(gerais[c])
    print(f"{c:25} {a:>18,.2f} {b:>18,.2f}  {'OK' if abs(a - b) < 0.005 else 'DIFERENTE'}")

In [0]:
kpis_por_ramo = calcular_kpis(titulos_dim, ["ramo_atividade", "ramo_descricao"])
kpis_por_uf = calcular_kpis(titulos_dim, ["uf"])
kpis_por_prazo = calcular_kpis(titulos_dim, ["cond_pagamento", "prazo_dias"])

kpis_por_ramo.select("ramo_atividade", "ramo_descricao", "qtd_clientes", "carteira_aberta", "pct_vencido",
                     "pct_pagos_com_atraso", "atraso_medio_ponderado", "dso", "dias_alem_prazo") \
             .orderBy(F.desc("pct_vencido")).show()
kpis_por_uf.select("uf", "qtd_clientes", "carteira_aberta", "pct_vencido",
                   "pct_pagos_com_atraso", "atraso_medio_ponderado", "dso", "dias_alem_prazo") \
           .orderBy(F.desc("pct_vencido")).show()
kpis_por_prazo.select("cond_pagamento", "prazo_dias", "qtd_clientes", "carteira_aberta", "pct_vencido",
                      "pct_pagos_com_atraso", "atraso_medio_ponderado", "dso", "dias_alem_prazo", "prazo_medio_ponderado") \
              .orderBy("prazo_dias").show()

In [0]:
def checar_somas(df, nome):
    print(nome)
    df.agg(
        F.sum("qtd_titulos").alias("qtd_titulos"),                  # 28725
        F.sum("carteira_aberta").alias("carteira_aberta"),          # 94812358.05
        F.sum("carteira_vencida").alias("carteira_vencida"),        # 20072845.68
        F.sum("qtd_pagos").alias("qtd_pagos"),                      # 25373
        F.sum("qtd_pagos_com_atraso").alias("qtd_pagos_com_atraso"),# 12147
        F.sum("faturamento_90d").alias("faturamento_90d"),          # 107103834.13
    ).show(vertical=True)

checar_somas(kpis_por_ramo, "por ramo")
checar_somas(kpis_por_uf, "por UF")
checar_somas(kpis_por_prazo, "por prazo")

In [0]:
abertos = titulos_dim.filter(F.col("status") == "aberto")
total_aberto = abertos.agg(F.sum("valor")).first()[0]

aging_carteira = (
    abertos.groupBy("faixa_aging")
    .agg(
        F.count(F.lit(1)).alias("qtd_titulos"),
        F.sum("valor").alias("valor_total"),
    )
    .withColumn("pct_carteira", F.round(F.col("valor_total") / F.lit(total_aberto) * 100, 2))
    .withColumn("ordem_faixa",                                      # CONFIRA: textos exatos das faixas
        F.when(F.col("faixa_aging") == "a vencer", 1)
         .when(F.col("faixa_aging") == "1-30", 2)
         .when(F.col("faixa_aging") == "31-60", 3)
         .when(F.col("faixa_aging") == "61-90", 4)
         .when(F.col("faixa_aging") == "90+", 5))
    .withColumn("data_corte", F.lit(DATA_CORTE).cast("date"))
)

aging_carteira.orderBy("ordem_faixa").show()

In [0]:
print("ordem nula:", aging_carteira.filter(F.col("ordem_faixa").isNull()).count())  # 0
aging_carteira.agg(
    F.sum("qtd_titulos").alias("qtd_titulos"),        # 3352
    F.sum("valor_total").alias("valor_total"),        # 94812358.05
    F.sum("pct_carteira").alias("soma_pct"),          # 100, com variação de 0,01 pelo arredondamento
).show(vertical=True)

In [0]:
salvar(kpis_por_ramo, GOLD, "kpis_por_ramo")
salvar(kpis_por_uf, GOLD, "kpis_por_uf")
salvar(kpis_por_prazo, GOLD, "kpis_por_prazo")
salvar(aging_carteira, GOLD, "aging_carteira")

In [0]:
spark.sql(f"""
    SELECT
        (SELECT COUNT(*) FROM {GOLD}.kpis_por_ramo)                AS linhas_ramo,
        (SELECT COUNT(*) FROM {GOLD}.kpis_por_uf)                  AS linhas_uf,
        (SELECT COUNT(*) FROM {GOLD}.kpis_por_prazo)               AS linhas_prazo,
        (SELECT COUNT(*) FROM {GOLD}.aging_carteira)               AS linhas_aging,
        (SELECT SUM(carteira_aberta) FROM {GOLD}.kpis_por_uf)      AS carteira_uf,
        (SELECT SUM(valor_total) FROM {GOLD}.aging_carteira)       AS carteira_aging
""").show(vertical=True)

In [0]:
vencidos_pe = (
    titulos_dim.filter(F.col("status") == "aberto")
    .filter(F.col("dias_atraso") > 0)
    .filter(F.col("uf") == "PE")
)

print(vencidos_pe.count())
vencidos_pe.agg(F.sum("valor")).show()

In [0]:
vencidos_pe_cliente = (
    vencidos_pe.groupBy("cod_cliente")
    .agg(F.count(F.lit(1)).alias("qtd_titulos"), F.sum("valor").alias("valor_vencido"), F.max("dias_atraso").alias("maior_atraso"))
)

vencidos_pe_cliente.orderBy(F.desc("valor_vencido")).show(10)

In [0]:
cliente_178 = (
    titulos_dim.filter(F.col("cod_cliente") == "0000100178")
    .groupBy("status")
    .agg(F.count(F.lit(1)).alias("qtd_titulos"), F.max("data_pagamento").alias("ultimo_pagamento"), F.max("data_documento").alias("ultima_fatura"))
)

cliente_178.show()

## Investigação: carteira vencida de Pernambuco

**Pergunta:** o recorte por UF mostrou PE muito acima dos demais estados (46,8 dias além do prazo e 40,45% da carteira vencida). O problema está espalhado entre os clientes ou concentrado em poucos?

**Resultado:**

- PE tem 104 títulos vencidos, somando R$ 3.623.590,94.
- Um único cliente (`0000100178`) concentra 79 desses títulos e R$ 2.946.302,56, ou seja, **81% do vencido do estado** e cerca de 15% de toda a carteira vencida da empresa.
- O cliente **continua ativo e pagando**: 170 títulos pagos, com o último pagamento em 10/09/2026. Os títulos vencidos se acumularam ao longo de quase dois anos (o mais antigo tem 673 dias de atraso).

**Conclusão:** o problema não é o estado nem o cliente como um todo, e sim títulos específicos que ficaram parados sem acompanhamento. Na prática, esse padrão costuma indicar títulos em disputa (avaria, devolução, diferença de preço), e a ação de cobrança é revisar título por título, e não mudar a política comercial para PE.

**Lições:**

- Uma média por segmento pode ser dominada por um único caso; antes de concluir, é preciso olhar a concentração.
- A hipótese inicial ("o cliente parou de pagar") foi testada e descartada pelos dados.
- O modelo de ML deve prever o risco **por título**, e não por cliente.

*Análise exploratória: estas células não gravam tabelas.*